# Text Preprocessing using the Coachella dataset

**Student:** Harutyun Abajyan
**Student ID:** 36111

The dataset has tweets about the Coachella 2015 music festival. In this notebook I:
1. extract all **hashtags** into a new column `hashtags`;
2. extract all **emails** into a new column `emails`;
3. write the 7 cleaning functions from the task (`remove_usernames`, `remove_links`, `remove_non_ascii_symbols`,
   `to_lower`, `remove_stop_words`, `remove_digits`, `remove_special_characters`) and apply them to a new
   tweet text column `clean_text`.

I use `re` (regular expressions) and `pandas`. The stop word list is written in the notebook, so nothing has to be
downloaded.

In [1]:
import re
from collections import Counter

import pandas as pd

pd.set_option("display.max_colwidth", 110)

## 1. Loading the data

The file is not valid UTF-8 (with the default encoding pandas stops with a `UnicodeDecodeError`), so I read it with
`latin-1`, which accepts every byte. The emojis in this dataset were already broken when the file was created
(they look like `_Ù_¦_Ù`), so they cannot be repaired. This is not a problem, because `remove_non_ascii_symbols`
removes them anyway.

In [2]:
df = pd.read_csv("Coachella-2015-2-DFE.csv", encoding="latin-1")
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df[["coachella_sentiment", "text"]].head(8)

Shape: (3846, 10)
Columns: ['coachella_sentiment', 'coachella_yn', 'name', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_id', 'tweet_location', 'user_timezone']


,coachella_sentiment,text
0,positive,#Coachella2015 tickets selling out in less than 40 minutes _Ù_¦_Ù___Ù___Ù÷_ÙÎµ_ÙÎµ_Ù___Ù_¦ http://t.co/Sm...
1,positive,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ´_ÙÓ´_ÙÓ´ #Coachella2015 http://t.co/Ov2eCJtAvR
2,positive,#Coachella2015 #VIP passes secured! See you there bitchesssss
3,positive,PhillyÛªs @warondrugsjams will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon: http:/...
4,positive,If briana and her mom out to #Coachella2015 im out with them !!! _Ù÷Ý_Ù÷Ý_Ù÷Ý_ÙÕÄ
5,positive,West side is the best side!\n#west #coas #Coachella2015 http://t.co/BhasNazMMU
6,positive,Coachella tickets are now sold out _Ù÷_ &amp; I had the opportunity to have bought them. I'm definitely go...
7,positive,#Coachella2015 I absolutely can NOT wait. This weekend is about to exceed EPIC EXPECTATIONS!!! :) Lineup i...


## 2. Hashtags

A hashtag is `#` followed by letters, digits or `_`. My pattern only accepts ASCII letters and digits, and the
hashtag has to **end with a letter or a digit**. The reason is that in this file some broken emoji characters are
glued directly to hashtags, for example `#Coachella2015_Ù`; the hashtag should be `#Coachella2015`.

In [3]:
HASHTAG_PATTERN = r"#[A-Za-z0-9_]*[A-Za-z0-9]"

def extract_hashtags(text):
    """Return a list with all hashtags of the tweet (in the order they appear)."""
    return re.findall(HASHTAG_PATTERN, text)


# small test with the examples from the task + a broken one from the dataset
print(extract_hashtags("So excited for #coachella and #beychella!! #Coachella2015_Ù"))

['#coachella', '#beychella', '#Coachella2015']


In [4]:
df["hashtags"] = df["text"].apply(extract_hashtags)
df[["text", "hashtags"]].head(8)

,text,hashtags
0,#Coachella2015 tickets selling out in less than 40 minutes _Ù_¦_Ù___Ù___Ù÷_ÙÎµ_ÙÎµ_Ù___Ù_¦ http://t.co/Sm...,[#Coachella2015]
1,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ´_ÙÓ´_ÙÓ´ #Coachella2015 http://t.co/Ov2eCJtAvR,[#Coachella2015]
2,#Coachella2015 #VIP passes secured! See you there bitchesssss,"[#Coachella2015, #VIP]"
3,PhillyÛªs @warondrugsjams will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon: http:/...,"[#Coachella2015, #GovBall2015]"
4,If briana and her mom out to #Coachella2015 im out with them !!! _Ù÷Ý_Ù÷Ý_Ù÷Ý_ÙÕÄ,[#Coachella2015]
5,West side is the best side!\n#west #coas #Coachella2015 http://t.co/BhasNazMMU,"[#west, #coas, #Coachella2015]"
6,Coachella tickets are now sold out _Ù÷_ &amp; I had the opportunity to have bought them. I'm definitely go...,"[#Coachella2015, #Coachella]"
7,#Coachella2015 I absolutely can NOT wait. This weekend is about to exceed EPIC EXPECTATIONS!!! :) Lineup i...,[#Coachella2015]


In [5]:
n_with_tag = (df["hashtags"].apply(len) > 0).sum()
all_tags = [tag.lower() for tags in df["hashtags"] for tag in tags]   # lower case: #Coachella = #coachella

print("Tweets with at least one hashtag:", n_with_tag, "of", len(df))
print("Hashtags found in total:", len(all_tags))
print("Different hashtags (ignoring upper/lower case):", len(set(all_tags)))
print()
print("10 most common hashtags:")
for tag, count in Counter(all_tags).most_common(10):
    print(f"  {tag:20} {count}")

Tweets with at least one hashtag: 3840 of 3846
Hashtags found in total: 5488
Different hashtags (ignoring upper/lower case): 833

10 most common hashtags:
  #coachella2015       3833
  #coachella           288
  #coachellalineup     66
  #drake               40
  #acdc                34
  #music               28
  #jackwhite           24
  #lineup              20
  #steelydan           17
  #coachellatickets    15


## 3. Emails

An email is: a name part (letters, digits and `. _ % + -`), then `@`, then a domain that contains a dot and ends
with at least 2 letters (`gmail.com`, `yahoo.co.uk`). Because of the domain part, a username like `@coachella` is
**not** taken as an email.

In [6]:
EMAIL_PATTERN = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

def extract_emails(text):
    """Return a list with all email addresses in the tweet."""
    return re.findall(EMAIL_PATTERN, text)


# test with the examples from the task
print(extract_emails("Send it to example@gmail.com or user@yahoo.com, not to @coachella"))

['example@gmail.com', 'user@yahoo.com']


In [7]:
df["emails"] = df["text"].apply(extract_emails)
print("Tweets with at least one email:", (df["emails"].apply(len) > 0).sum())
df[["text", "emails"]].head(5)

Tweets with at least one email: 0


,text,emails
0,#Coachella2015 tickets selling out in less than 40 minutes _Ù_¦_Ù___Ù___Ù÷_ÙÎµ_ÙÎµ_Ù___Ù_¦ http://t.co/Sm...,[]
1,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ´_ÙÓ´_ÙÓ´ #Coachella2015 http://t.co/Ov2eCJtAvR,[]
2,#Coachella2015 #VIP passes secured! See you there bitchesssss,[]
3,PhillyÛªs @warondrugsjams will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon: http:/...,[]
4,If briana and her mom out to #Coachella2015 im out with them !!! _Ù÷Ý_Ù÷Ý_Ù÷Ý_ÙÕÄ,[]


The result is **0**: there is no email address in this dataset, so the `emails` column contains an empty list
for every tweet. The function itself works (see the test above). To be sure that I did not miss anything, I
checked what the words with an `@` in the tweets look like: they are all usernames.

In [8]:
at_words = [word for text in df["text"] for word in text.split() if "@" in word]
print("Words that contain '@':", len(at_words))
print("Most common:", Counter(at_words).most_common(8))
print("Words with '@' that also have a '.' after the '@':",
      [w for w in at_words if "." in w.split("@", 1)[1].rstrip(".:,!?")])

Words that contain '@': 2349
Most common: [('@coachella', 288), ('@Drake', 89), ('\x89ÛÏ@Forever21:', 42), ('@theweeknd', 42), ('@acdc', 26), ('@AZEALIABANKS', 21), ('@', 19), ('@Coachella', 18)]
Words with '@' that also have a '.' after the '@': []


## 4. Cleaning functions

Every function takes a string and returns a string. When a function removes something, it puts a **space** in its
place (so the words on both sides are not glued together), and at the end multiple spaces are squeezed into one
with `" ".join(text.split())`.

In [9]:
def squeeze_spaces(text):
    """Replace any run of whitespace (also new lines) with a single space."""
    return " ".join(text.split())


def remove_usernames(text):
    """Remove usernames like @coachella or @beyonce.
    A username is @ followed by letters, digits or _.
    (?<![A-Za-z0-9._%+-]) means: the @ must not come right after a letter/digit,
    so the @ inside an email like user@yahoo.com is not treated as a username."""
    text = re.sub(r"(?<![A-Za-z0-9._%+-])@[A-Za-z0-9_]+", " ", text)
    return squeeze_spaces(text)


def remove_links(text):
    """Remove links that start with http or https: everything from 'http' to the next space.
    This also catches links that Twitter cut off at the end of a tweet (like 'http...')."""
    text = re.sub(r"http\S*", " ", text)
    return squeeze_spaces(text)


def remove_non_ascii_symbols(text):
    """Remove every character outside the ASCII range (code 0-127): emojis, accented letters, etc."""
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    return squeeze_spaces(text)


def to_lower(text):
    """Convert all letters to lower case."""
    return text.lower()


def remove_digits(text):
    """Remove all numbers (2018, 10, 100 ...). The digits inside a word are removed too: coachella2015 -> coachella."""
    text = re.sub(r"\d+", "", text)
    return squeeze_spaces(text)


def remove_special_characters(text):
    """Remove punctuation and other special characters: . , ! ? # $ % & * ( ) + = - _ [ ] { } ; : ' " / \\ | < > ` ~
    Twitter saves &, < and > as HTML codes (&amp; &lt; &gt;). I remove these codes as a whole first,
    otherwise '&amp;' would leave the word 'amp' in the text."""
    text = re.sub(r"&[A-Za-z]+;", " ", text)
    # everything that is not a letter, a digit or whitespace -> space
    # (\w counts "_" as a letter, so I add it separately)
    text = re.sub(r"[^\w\s]|_", " ", text)
    return squeeze_spaces(text)

### Stop words

I use the English stop word list of NLTK (`nltk.corpus.stopwords.words("english")`, NLTK 3.10, 198 words),
written out here. Because my pipeline removes the apostrophes before the stop words (see below), I also add every
word without its apostrophe (`"don't"` -> `"dont"`, `"i'm"` -> `"im"`), since people often write it like that in
tweets anyway. Finally I add a few Twitter words with no meaning: `rt` (retweet), `u`, `ur`.

In [10]:
NLTK_STOP_WORDS = """
a about above after again against ain all am an and any are aren aren't as at be because been before being below
between both but by can couldn couldn't d did didn didn't do does doesn doesn't doing don don't down during each
few for from further had hadn hadn't has hasn hasn't have haven haven't having he he'd he'll her here hers herself
he's him himself his how i i'd if i'll i'm in into is isn isn't it it'd it'll it's its itself i've just ll m ma me
mightn mightn't more most mustn mustn't my myself needn needn't no nor not now o of off on once only or other our
ours ourselves out over own re s same shan shan't she she'd she'll she's should shouldn shouldn't should've so some
such t than that that'll the their theirs them themselves then there these they they'd they'll they're they've
this those through to too under until up ve very was wasn wasn't we we'd we'll we're were weren weren't we've what
when where which while who whom why will with won won't wouldn wouldn't y you you'd you'll your you're yours
yourself yourselves you've
""".split()

STOP_WORDS = set(NLTK_STOP_WORDS)
STOP_WORDS |= {word.replace("'", "") for word in NLTK_STOP_WORDS}   # dont, im, youre, ...
STOP_WORDS |= {"rt", "u", "ur"}                                     # Twitter words
print("NLTK list:", len(NLTK_STOP_WORDS), "words | my final list:", len(STOP_WORDS), "words")


def remove_stop_words(text):
    """Remove common words that do not add meaning (the, a, an, and, ...)."""
    words = [word for word in text.split() if word not in STOP_WORDS]
    return " ".join(words)

NLTK list: 198 words | my final list: 244 words


### Testing each function on the examples from the task

In [11]:
print(remove_usernames("Can't wait to see @beyonce at @coachella!"))
print(remove_usernames("my email example@gmail.com stays, @coachella goes"))
print(remove_links("Tickets: https://www.coachella.com/ or http://bit.ly/2GdOZLz now"))
print(remove_non_ascii_symbols("Beyoncé was amazing 🔥🔥 at Coachella"))
print(to_lower("COACHELLA was AMAZING, Beyoncé!"))
print(remove_stop_words("the lineup is an amazing mix of the best artists and a lot of fun"))
print(remove_digits("Coachella 2018 had 100 artists in 10 days"))
print(remove_special_characters("Wow!!! #coachella (VIP) $500 & more... \"great\" / [yes] {ok} ~_~ &amp; me"))

Can't wait to see at !
my email example@gmail.com stays, goes
Tickets: or now
Beyonc was amazing at Coachella
coachella was amazing, beyoncé!
lineup amazing mix best artists lot fun
Coachella had artists in days
Wow coachella VIP 500 more great yes ok me


## 5. Applying the functions to the new tweet column

First I create the new column `clean_text` as a copy of `text`, then I apply the functions to it one after
another. The order matters:

* **usernames and links first**, while they are still complete. Later steps would break them into pieces
  (`http://t.co/abc` would become `http t co abc` after removing special characters and could not be found anymore).
* **non-ASCII symbols** next, then **lower case**, **digits** and **special characters**.
* **stop words last.** The task lists `remove_stop_words` before `remove_digits` and `remove_special_characters`,
  but I moved it to the end: the stop words are compared word by word, and before the punctuation is removed a word
  like `"the,"` or `"I'm"` would not match `"the"` / `"i"` + `"m"`, so many stop words would stay in the text.

To show what every step does, I print the same example tweet after each step.

In [12]:
cleaning_steps = [
    remove_usernames,
    remove_links,
    remove_non_ascii_symbols,
    to_lower,
    remove_digits,
    remove_special_characters,
    remove_stop_words,
]

df["clean_text"] = df["text"]     # the new tweets text column
example = 3                       # this tweet has a broken emoji, a username, hashtags, &amp;, digits and links

print(f"{'original':26} | {df.loc[example, 'text']!r}")
for step in cleaning_steps:
    df["clean_text"] = df["clean_text"].apply(step)
    print(f"{step.__name__:26} | {df.loc[example, 'clean_text']!r}")

original                   | 'Philly\x89Ûªs @warondrugsjams will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon: http://t.co/wzaAE5wYNT http://t.co/sFL9I5sD20'
remove_usernames           | 'Philly\x89Ûªs will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon: http://t.co/wzaAE5wYNT http://t.co/sFL9I5sD20'
remove_links               | 'Philly\x89Ûªs will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon:'
remove_non_ascii_symbols   | 'Philly s will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon:'
to_lower                   | 'philly s will play #coachella2015 &amp; #govball2015! watch them on jimmy fallon:'
remove_digits              | 'philly s will play #coachella &amp; #govball! watch them on jimmy fallon:'
remove_special_characters  | 'philly s will play coachella govball watch them on jimmy fallon'
remove_stop_words          | 'philly play coachella govball watch jimmy fallon'


In [13]:
df[["text", "clean_text"]].head(10)

,text,clean_text
0,#Coachella2015 tickets selling out in less than 40 minutes _Ù_¦_Ù___Ù___Ù÷_ÙÎµ_ÙÎµ_Ù___Ù_¦ http://t.co/Sm...,coachella tickets selling less minutes
1,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ´_ÙÓ´_ÙÓ´ #Coachella2015 http://t.co/Ov2eCJtAvR,wait absolute fire coachella
2,#Coachella2015 #VIP passes secured! See you there bitchesssss,coachella vip passes secured see bitchesssss
3,PhillyÛªs @warondrugsjams will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon: http:/...,philly play coachella govball watch jimmy fallon
4,If briana and her mom out to #Coachella2015 im out with them !!! _Ù÷Ý_Ù÷Ý_Ù÷Ý_ÙÕÄ,briana mom coachella
5,West side is the best side!\n#west #coas #Coachella2015 http://t.co/BhasNazMMU,west side best side west coas coachella
6,Coachella tickets are now sold out _Ù÷_ &amp; I had the opportunity to have bought them. I'm definitely go...,coachella tickets sold opportunity bought definitely going next year coachella coachella
7,#Coachella2015 I absolutely can NOT wait. This weekend is about to exceed EPIC EXPECTATIONS!!! :) Lineup i...,coachella absolutely wait weekend exceed epic expectations lineup crazy
8,If someone got me to Coachella if be your friend for life #truth #desprate #Coachella2015 #Coachella http:...,someone got coachella friend life truth desprate coachella coachella
9,RT @brownjenjen:  Õ http://t.co/mxCREvIlGP 712 Coachella 2015 Coachella 2015 makes space for rockers and...,coachella coachella makes space rockers ravers year coachella oh


### Checking the result

To make sure the cleaning worked on the whole column, I count how many cleaned tweets still contain each unwanted
element. All counts should be 0.

In [14]:
clean = df["clean_text"]
checks = {
    "a username (@...)":        clean.str.contains(r"@\w", regex=True).sum(),
    "a link (http...)":         clean.str.contains("http", regex=False).sum(),
    "a non-ASCII character":    clean.str.contains(r"[^\x00-\x7F]", regex=True).sum(),
    "an upper case letter":     clean.str.contains(r"[A-Z]", regex=True).sum(),
    "a digit":                  clean.str.contains(r"\d", regex=True).sum(),
    "a special character":      clean.str.contains(r"[^a-z ]", regex=True).sum(),
    "a stop word":              clean.apply(lambda t: any(w in STOP_WORDS for w in t.split())).sum(),
}
for name, count in checks.items():
    print(f"Cleaned tweets that still contain {name:24} {count}")

Cleaned tweets that still contain a username (@...)        0
Cleaned tweets that still contain a link (http...)         0
Cleaned tweets that still contain a non-ASCII character    0
Cleaned tweets that still contain an upper case letter     0
Cleaned tweets that still contain a digit                  0
Cleaned tweets that still contain a special character      0
Cleaned tweets that still contain a stop word              0


In [15]:
words_before = df["text"].apply(lambda t: len(t.split()))
words_after = df["clean_text"].apply(lambda t: len(t.split()))

print(f"Average words per tweet: {words_before.mean():.2f} before -> {words_after.mean():.2f} after cleaning")
print("Tweets that are empty after cleaning:", (words_after == 0).sum())
print()
print("15 most common words after cleaning:")
word_counts = Counter(word for text in df["clean_text"] for word in text.split())
for word, count in word_counts.most_common(15):
    print(f"  {word:15} {count}")

Average words per tweet: 12.08 before -> 6.36 after cleaning
Tweets that are empty after cleaning: 0

15 most common words after cleaning:
  coachella       4659
  lineup          695
  go              427
  line            350
  year            280
  see             265
  going           249
  drake           220
  tickets         176
  like            160
  dc              147
  ac              146
  weekend         145
  excited         120
  wait            119


The most common word is `coachella`, because almost every tweet has `#Coachella2015` and the cleaning turns it into
`coachella` (the `#` is a special character and `2015` are digits). The original hashtags are not lost, they are
saved in the `hashtags` column. The words `ac` and `dc` come from the band name "AC/DC": the `/` is a special
character, so the name is split into two words. This is a typical side effect of removing punctuation.

## 6. Saving the result

I save the table with the new columns. In the CSV file the hashtag and email lists are written as text separated by
spaces.

In [16]:
output = df.copy()
output["hashtags"] = output["hashtags"].apply(" ".join)
output["emails"] = output["emails"].apply(" ".join)
output.to_csv("coachella_cleaned.csv", index=False)

print("Saved coachella_cleaned.csv with", output.shape[0], "rows and these columns:")
print(list(output.columns))
output[["text", "hashtags", "emails", "clean_text"]].head(5)

Saved coachella_cleaned.csv with 3846 rows and these columns:
['coachella_sentiment', 'coachella_yn', 'name', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_id', 'tweet_location', 'user_timezone', 'hashtags', 'emails', 'clean_text']


,text,hashtags,emails,clean_text
0,#Coachella2015 tickets selling out in less than 40 minutes _Ù_¦_Ù___Ù___Ù÷_ÙÎµ_ÙÎµ_Ù___Ù_¦ http://t.co/Sm...,#Coachella2015,,coachella tickets selling less minutes
1,RT @sudsybuddy: WAIT THIS IS ABSOLUTE FIRE _ÙÓ´_ÙÓ´_ÙÓ´ #Coachella2015 http://t.co/Ov2eCJtAvR,#Coachella2015,,wait absolute fire coachella
2,#Coachella2015 #VIP passes secured! See you there bitchesssss,#Coachella2015 #VIP,,coachella vip passes secured see bitchesssss
3,PhillyÛªs @warondrugsjams will play #Coachella2015 &amp; #GovBall2015! Watch them on Jimmy Fallon: http:/...,#Coachella2015 #GovBall2015,,philly play coachella govball watch jimmy fallon
4,If briana and her mom out to #Coachella2015 im out with them !!! _Ù÷Ý_Ù÷Ý_Ù÷Ý_ÙÕÄ,#Coachella2015,,briana mom coachella


## Summary

* `hashtags`: 3840 of the 3846 tweets have at least one hashtag; 5488 hashtags in total, 833 different ones
  (ignoring upper/lower case). The most common one is `#coachella2015`.
* `emails`: there are no email addresses in this dataset, so the column is empty for all tweets; the function was
  tested on example sentences and works.
* The 7 cleaning functions are applied to the new column `clean_text`. The check above shows that no username,
  link, non-ASCII character, upper case letter, digit, special character or stop word is left.
* I changed one thing compared to the order in the task: stop words are removed at the end, after the punctuation,
  so that words like `"The,"` or `"I'm"` are also recognised as stop words.
* Small extra fixes for this dataset: broken emoji characters glued to hashtags are not taken as part of the
  hashtag, cut-off links (`http...`) are removed too, and the HTML codes `&amp;`, `&lt;`, `&gt;` do not leave words
  like `amp` in the text.